# 05 - Data preprocessing

Objetivo: comparar escalamiento robusto, estandar y PowerTransformer sin incluir identificadores.

In [ ]:
import sys
from pathlib import Path
import polars as pl
sys.path.insert(0, str(Path.cwd().parent))
from analysis.pipeline import load_transactions, build_customer_features, transform_features

ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
features = build_customer_features(load_transactions(ROOT / "transacciones_sinteticas.csv"))
matrices = {}
for scaler in ("robust", "standard", "power"):
    matrices[scaler], columns = transform_features(features, scaler=scaler)
    print(scaler, matrices[scaler].shape, "id_included=", "codigo_cliente" in columns)

In [ ]:
summary = []
for scaler, matrix in matrices.items():
    summary.append({"scaler": scaler, "mean_abs": float(abs(matrix.mean(axis=0)).mean()), "std_mean": float(matrix.std(axis=0).mean()), "max_abs": float(abs(matrix).max())})
preprocessing_comparison = pl.DataFrame(summary)
preprocessing_comparison.write_csv(ROOT / "results" / "metrics" / "preprocessing_comparison.csv")
preprocessing_comparison

## Decision
El escalamiento robusto se mantiene como estrategia principal por la presencia documentada de colas largas; las alternativas quedan registradas para comparar estabilidad.